# 01 - Visual Exploration of HICP Eurozone

Initial visual inspection of the Harmonised Index of Consumer Prices (HICP) for the Euro Area.

**Input:** `data/processed/hicp_europe_index.parquet`  
**Series:** M.U2.N.000000.4.INX (ECB SDMX) - base 2015 = 100  
**Establishes:** Series range, trend, seasonal pattern, train/val/test splits

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path('..').resolve()         # tfg-forecasting/
MONOREPO = ROOT.parent              # tfg-ipc-mcp/
sys.path.insert(0, str(MONOREPO))

from shared.constants import DATE_TRAIN_END, DATE_VAL_END, FREQ

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

## 1. Data Loading

In [ ]:
DATA_PATH = ROOT / "data" / "processed" / "hicp_europe_index.parquet"
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df.index.freq = "MS"
print(f"Range: {df.index.min().date()} -> {df.index.max().date()}")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.describe().T

In [ ]:
# Verify frequency and gaps
expected = pd.date_range(df.index.min(), df.index.max(), freq="MS")
missing = expected.difference(df.index)
print(f"Expected months: {len(expected)}")
print(f"Present months: {len(df)}")
print(f"Gaps: {len(missing)}")
if len(missing) > 0:
    print("Missing dates:", missing.tolist())

## 2. Full Series - Index Level

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["hicp_index"], linewidth=1.2, color="#1565c0")

for label, date, color in [
    ("Train end", DATE_TRAIN_END, "green"),
    ("Val end", DATE_VAL_END, "orange"),
]:
    ax.axvline(pd.Timestamp(date), color=color, linestyle="--", alpha=0.7, label=label)

ax.axhline(100, color="#888", linewidth=0.8, linestyle=":", label="Base 2015=100")
ax.set_title("HICP Eurozone - Index Level (base 2015=100)")
ax.set_ylabel("HICP Index")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 3. Year-on-Year Change (YoY %)

In [ ]:
yoy = df["hicp_index"].pct_change(12) * 100

fig, ax = plt.subplots()
ax.plot(yoy.index, yoy, linewidth=1, color="#d62728")
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(2, color="grey", linewidth=0.5, linestyle=":", label="ECB Target (2%)")
ax.fill_between(yoy.index, yoy, 2, where=(yoy > 2), alpha=0.15, color="#d62728", label="Above target")
ax.fill_between(yoy.index, yoy, 2, where=(yoy < 2), alpha=0.15, color="#1565c0", label="Below target")

ax.set_title("HICP Eurozone - Year-on-Year Change (%)")
ax.set_ylabel("%")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 4. Monthly Change (MoM %) and Statistics by Month

In [ ]:
mom = df["hicp_index"].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(mom.index, mom, linewidth=0.9, color="#e65100")
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("Monthly Change (MoM %)")
axes[0].set_ylabel("%")
axes[0].xaxis.set_major_locator(mdates.YearLocator(2))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

monthly_df = pd.DataFrame({"month": df.index.month, "change_pct": mom})
monthly_df.boxplot(column="change_pct", by="month", ax=axes[1])
axes[1].set_title("Monthly Change Distribution (MoM %) by month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Monthly Change (%)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Moving Averages - Long-term Trend

In [ ]:
y = df["hicp_index"]
ma12 = y.rolling(12).mean()
ma24 = y.rolling(24).mean()

fig, ax = plt.subplots()
ax.plot(y.index, y, linewidth=1, color="#1565c0", alpha=0.5, label="HICP level")
ax.plot(ma12.index, ma12, linewidth=2, color="#e65100", label="MA 12m")
ax.plot(ma24.index, ma24, linewidth=2, color="#2e7d32", label="MA 24m")
ax.axhline(100, color="#888", linewidth=0.8, linestyle=":")
ax.set_title("HICP Eurozone - Trend via Moving Averages")
ax.set_ylabel("HICP Index")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 6. Train / Validation / Test Splits

In [ ]:
train = df.loc[:DATE_TRAIN_END]
val   = df.loc[DATE_TRAIN_END:DATE_VAL_END].iloc[1:]
test  = df.loc[DATE_VAL_END:].iloc[1:]

fig, ax = plt.subplots()
ax.plot(train.index, train["hicp_index"], label=f"Train ({len(train)})", color="#2ca02c")
ax.plot(val.index,   val["hicp_index"],   label=f"Val ({len(val)})",   color="#ff7f0e")
ax.plot(test.index,  test["hicp_index"],  label=f"Test ({len(test)})",  color="#d62728")

ax.set_title("HICP Eurozone - Temporal Splits")
ax.set_ylabel("HICP Index")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

print(f"Train: {train.index.min().date()} -> {train.index.max().date()} ({len(train)} months)")
print(f"Val:   {val.index.min().date()} -> {val.index.max().date()} ({len(val)} months)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()} ({len(test)} months)")

## 7. Summary

In [ ]:
yoy_full = df["hicp_index"].pct_change(12) * 100
print("="*60)
print("HICP EUROZONE - EXPLORATION SUMMARY")
print("="*60)
print(f"Temporal range:   {df.index.min().date()} -> {df.index.max().date()}")
print(f"Total months:     {len(df)}")
print(f"Frequency:        Monthly (MS)")
print(f"Gaps:             {len(missing)}")
print(f"NaN:              {df['hicp_index'].isna().sum()}")
print(f"Min index:        {df['hicp_index'].min():.2f} ({df['hicp_index'].idxmin().date()})")
print(f"Max index:        {df['hicp_index'].max():.2f} ({df['hicp_index'].idxmax().date()})")
print(f"Max YoY (%):      {yoy_full.max():.2f}% ({yoy_full.idxmax().date()})")
print(f"Min YoY (%):      {yoy_full.min():.2f}% ({yoy_full.idxmin().date()})")
print("="*60)